Testar att beräkna avstånd mellan två adresser med hjälp av geopy.    
Tittar på följande Youtube-film för att förstå konceptet: https://www.youtube.com/watch?v=W4VArWmB7hI

Inser dock, efter test, att avståndet räknas fågelvägen om jag följer den kod som visas i videon, vilket inte är så bra om man ska göra en taxi-prediktion...

In [1]:
from geopy.geocoders import Nominatim
from geopy import distance

In [2]:

geolocator = Nominatim(user_agent="hid-training")

address1 = "Rangströmsliden 3, 41318, Göteborg, SE"
address2 = "A C Lindblads Gata 10, 41871, Göteborg, SE "


In [ ]:
location1 = geolocator.geocode(address1)
location2 = geolocator.geocode(address2)

print(location1)
print(location2)

In [4]:
coord1 = (location1.latitude, location1.longitude)
coord2 = (location2.latitude, location2.longitude)

In [5]:
#Geo Disc Distance
distance_default = distance.distance(coord1, coord2)

print(distance_default.km)

3.345388248120942


In [6]:
#Great Circle Distance
distance_great_circle = distance.great_circle(coord1, coord2)

print(distance_great_circle.km)

3.3375481781539245


Undersöker om OSMR kan vara aktuellt att använda för att beräkna avstånd mellan två olika adresser.    
Använder mig av följande YouTube-film: https://www.youtube.com/watch?v=Ke_NISW-bDM   
Börjar med att installera Docker för att kunna använda OSMR

In [ ]:
#Test för att se att docker-servern funkar. 

import requests
r = requests.get("http://localhost:5000/route/v1/driving/11.973,57.708;12.001,57.697", params={"overview":"false"})
r.raise_for_status()
data = r.json()
print(data["routes"][0]["distance"]/1000, "km", " | ", data["routes"][0]["duration"]/60, "min")


In [ ]:
def geocode(address: str) -> tuple[float, float]:
    """
    Geokoda en adress till (lat, lon) via Nominatim/OSM.
    Tips: sätt en unik user_agent för att vara snäll mot tjänsten.
    """
    geoloc = Nominatim(user_agent="taxipred-notebook/0.1")
    loc = geoloc.geocode(address, timeout=10)
    if not loc:
        raise ValueError(f"Kunde inte geokoda: {address}")
    return float(loc.latitude), float(loc.longitude)

In [ ]:
a = "Rangströmsliden 3, Göteborg"
b = "Liseberg, Göteborg"
lat1, lon1 = geocode(a)
lat2, lon2 = geocode(b)
(lat1, lon1), (lat2, lon2)

((57.6970804, 11.9432269), (57.6945173, 11.9936954))

In [ ]:
import requests

OSRM_BASE = "http://localhost:5000"

def route_distance_km_osrm(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """
    Hämtar ruttdistans (km) med OSRM 'driving'-profil.
    Returnerar kilometer. Kräver att osrm-routed kör lokalt.
    """
    url = f"{OSRM_BASE}/route/v1/driving/{lon1},{lat1};{lon2},{lat2}"
    params = {"overview": "false", "alternatives": "false", "steps": "false"}
    r = requests.get(url, params=params, timeout=10)
    r.raise_for_status()
    data = r.json()
    if not data.get("routes"):
        raise RuntimeError("Ingen rutt hittades av OSRM")
    meters = data["routes"][0]["distance"]
    return meters / 1000.0

# Snabbtest:
route_distance_km_osrm(lat1, lon1, lat2, lon2)


4.8103

Jag har försökt att få Docker att fungera, för att kunna använda OSMR för avståndsberäkning.   
Visar sig dock vara krångligt att få det att funka, och jag undersöker därför fler sätt att kunna göra avståndsberäkningar.   
Använder Gemini, som tipsar om  Google Maps API, vilket verkar vara en enkare lösning.    
Testar därför hur det funkar att använda Distance Matrix API, för att se om det är bästa vägen framåt för att få en fungerande avståndsberäkning. 


In [26]:
import googlemaps
import toml
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

print(PROJECT_ROOT)

c:\Users\susan\KodPython\LAB_taxi_pred


In [27]:
SECRETS_FILE = PROJECT_ROOT /'secrets.toml'

print(SECRETS_FILE)

c:\Users\susan\KodPython\LAB_taxi_pred\secrets.toml


In [ ]:
secrets = toml.loads(SECRETS_FILE.read_text())
GOOGLE_API_KEY = secrets['google_api_key']

#print(GOOGLE_API_KEY)


In [30]:
gmaps = googlemaps.Client(key=GOOGLE_API_KEY)

In [31]:
ORIGIN = "Kungsportsplatsen, Göteborg"
DESTINATION = "Landvetter Flygplats, Göteborg"

In [32]:

print(f"Bräknar avståndet mellan {ORIGIN} och {DESTINATION}")

Bräknar avståndet mellan Kungsportsplatsen, Göteborg och Landvetter Flygplats, Göteborg


In [ ]:
try:
    result = gmaps.distance_matrix(
        origins = [ORIGIN],
        destinations = [DESTINATION],
        mode = "driving",
        language = "sv"
    )

    element = result['rows'][0]['elements'][0]

    if element['status'] == 'OK':
        distance_text = element['distance']['text']
        distance_km = element['distance']['value'] /1000.0

        duration_text = element['duration']['text']
        duration_minutes = element['duration']['value'] / 60.0

        print("=========== RESULTAT =============")
        print("Status = OK")
        print(f"AVSTÅND (text): {distance_text}")
        print(f"AVSTÅND (km): {distance_km:.2f} km")
        print(f"RESTID (text): {duration_text}")
        print(f"RESTID (min): {duration_minutes:.0f} minuter")
        print("==================================")
    
    else:
        print(f"\nFEL! Rutt kunde inte hittas!. Status: {element['status']}")
        print("Kontrollera stavningen på adresserna.")

except Exception as e:
    print(f"\nKRITISKT FEL: Ett fel uppstod vid kommunikation med API:et.")
    print(f"Kontrollera din API-nyckel och internetanslutning. Fel: {e}")

    

=========== RESULTAT =============
Status = OK
Avstånd (text): 25,3 km
Avstånd (km): 25.35 km
Restid (text): 23 min
Restid (min): 23 minuter
